5. Creación y entrenamiento del modelo

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import time

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor

import warnings
warnings.filterwarnings('ignore')

In [ ]:
CSV_PATH = "../datasets/processed/sevilla_dataset_preparado.csv"

df = pd.read_csv(CSV_PATH)

print(f"Dataset cargado correctamente: {CSV_PATH}")
print(f"Total de píxeles (registros): {len(df)}")
print(f"Variables disponibles: {list(df.columns)}")

df.head()

✅ Dataset cargado correctamente: ../datasets/processed/sevilla_dataset_preparado.csv
Total de píxeles (registros): 484356
Variables disponibles: ['Longitude', 'Latitude', 'NDVI', 'NDBI', 'Albedo', 'D2W_meters', 'LST_Target', 'D2R_HighCapacity_m', 'D2R_Urban_m', 'Tree_Density_50m', 'Building_Density_100m', 'Avg_Building_Height_100m']


,Longitude,Latitude,NDVI,NDBI,Albedo,D2W_meters,LST_Target,D2R_HighCapacity_m,D2R_Urban_m,Tree_Density_50m,Building_Density_100m,Avg_Building_Height_100m
0,-6.029941,37.449956,0.505166,-0.178238,0.157998,20.000000,40.960874,2743.999249,2776.807410,1,0,0.0
1,-6.029762,37.449956,0.728687,-0.395285,0.181745,20.000000,40.714777,2743.879563,2760.939843,1,0,0.0
2,-6.029582,37.449956,0.763079,-0.424442,0.149877,20.000000,40.714777,2743.851804,2745.072446,1,0,0.0
3,-6.029402,37.449956,0.775685,-0.418584,0.147389,22.360680,40.379811,2743.915975,2729.205220,1,0,0.0
4,-6.029223,37.449956,0.835151,-0.530323,0.201774,28.284271,40.379811,2744.072070,2713.338171,1,0,0.0


In [ ]:
# 1. Definimos cuáles son las variables predictoras (Features)
# Excluimos Lat y Lon para que el modelo aprenda de física y urbanismo, no de coordenadas
features = [
    'NDVI', 
    'NDBI', 
    'Albedo',
    'D2W_meters', 
    'D2R_HighCapacity_m',
    'D2R_Urban_m',
    'Tree_Density_50m',
    'Building_Density_100m',
    'Avg_Building_Height_100m'
]

target = 'LST_Target'

# Eliminamos cualquier posible valor nulo que se haya colado en el CSV
df_clean = df.dropna(subset=features + [target]).copy()

X = df_clean[features] 
y = df_clean[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Escalamos los datos (Crucial para modelos lineales y redes neuronales)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Tamaño de Entrenamiento: {X_train.shape}")
print(f"Tamaño de Test: {X_test.shape}")

In [ ]:
# Definimos 4 modelos competidores
modelos = {
    "Ridge Regression": Ridge(alpha=1.0),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    "XGBoost": XGBRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    "LightGBM": LGBMRegressor(n_estimators=100, random_state=42, n_jobs=-1, verbose=-1)
}

resultados = []

print("Evaluando modelos con Cross-Validation (5 particiones)...")
for nombre, modelo in modelos.items():
    # Usamos R2 y RMSE como métricas
    cv_scores_r2 = cross_val_score(modelo, X_train_scaled, y_train, cv=5, scoring='r2')
    cv_scores_mse = cross_val_score(modelo, X_train_scaled, y_train, cv=5, scoring='neg_mean_squared_error')
    rmse_scores = np.sqrt(-cv_scores_mse)
    
    resultados.append({
        "Modelo": nombre,
        "R2 Promedio": cv_scores_r2.mean(),
        "RMSE Promedio (°C)": rmse_scores.mean()
    })

df_resultados = pd.DataFrame(resultados).sort_values(by="R2 Promedio", ascending=False)
display(df_resultados)

In [ ]:
# El ganador de la Fase 1: Random Forest
print("🌳 Iniciando Grid Search para Random Forest (Versión Segura y Ligera)...")

# Definimos el modelo base
modelo_ganador = RandomForestRegressor(random_state=42, n_jobs=-1)

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [15, 20, 25], 
    'min_samples_split': [10, 20, 50],
    'min_samples_leaf': [10, 20],
    'max_features': ['sqrt', 'log2'] 
}

grid_search = GridSearchCV(
    estimator=modelo_ganador,
    param_grid=param_grid,
    cv=3,
    scoring='r2',
    n_jobs=-1,                              
    verbose=2                               
)

grid_search.fit(X_train_scaled, y_train)

print(f"\nMejores hiperparámetros encontrados: {grid_search.best_params_}")
mejor_modelo_final_RandomForest = grid_search.best_estimator_

In [ ]:
y_pred = mejor_modelo_final_RandomForest.predict(X_test_scaled)

print("--- MÉTRICAS FINALES EN TEST ---")
print(f"R2 Score: {r2_score(y_test, y_pred):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.4f} °C")
print(f"MAE: {mean_absolute_error(y_test, y_pred):.4f} °C")

importancias = mejor_modelo_final_RandomForest.feature_importances_
nombres_variables = X.columns

plt.figure(figsize=(10, 6))
sns.barplot(x=importancias, y=nombres_variables, palette="viridis")
plt.title("Importancia de las Variables en la Temperatura Superficial")
plt.xlabel("Peso en la decisión del modelo")
plt.show()

In [ ]:
os.makedirs("models", exist_ok=True)

ruta_modelo = "../models/rf_sevilla_temperatura_v1.pkl"

# Guardamos el modelo
joblib.dump(mejor_modelo_final_RandomForest, ruta_modelo)

print(f"✅ ¡Modelo guardado de forma segura en {ruta_modelo}!")

In [ ]:
print("Iniciando Grid Search para el modelo ganador...")

modelo_ganador = XGBRegressor(random_state=42, verbose=-1)

param_grid = {
    'n_estimators': [100, 300, 500],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [5, 10, -1],
    'num_leaves': [31, 63, 127]
}

grid_search = GridSearchCV(
    estimator=modelo_ganador,
    param_grid=param_grid,
    cv=3,
    scoring='r2',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train_scaled, y_train)

print(f"\nMejores hiperparámetros encontrados: {grid_search.best_params_}")
mejor_modelo_final_XGBoost = grid_search.best_estimator_

In [ ]:
y_pred = mejor_modelo_final_XGBoost.predict(X_test_scaled)

print("--- MÉTRICAS FINALES EN TEST ---")
print(f"R2 Score: {r2_score(y_test, y_pred):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.4f} °C")
print(f"MAE: {mean_absolute_error(y_test, y_pred):.4f} °C")

importancias = mejor_modelo_final_XGBoost.feature_importances_
nombres_variables = X.columns

plt.figure(figsize=(10, 6))
sns.barplot(x=importancias, y=nombres_variables, palette="viridis")
plt.title("Importancia de las Variables en la Temperatura Superficial")
plt.xlabel("Peso en la decisión del modelo")
plt.show()

In [ ]:
os.makedirs("models", exist_ok=True)

# Ruta donde guardaremos el modelo ganador
ruta_modelo = "../models/xgboost_sevilla_temperatura_v1.pkl"

# Guardamos el modelo
joblib.dump(mejor_modelo_final_XGBoost, ruta_modelo)

print(f"¡Modelo guardado de forma segura en {ruta_modelo}!")

In [ ]:
inicio = time.time()
xgb_preds = mejor_modelo_final_XGBoost.predict(X_test_scaled)
fin = time.time()
print(f"XGBoost tardó: {fin - inicio:.4f} segundos")

inicio = time.time()
rf_preds = mejor_modelo_final_RandomForest.predict(X_test_scaled)
fin = time.time()
print(f"Random Forest tardó: {fin - inicio:.4f} segundos")

In [ ]:
sns.set_theme(style="whitegrid")

fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)
min_val = min(y_test.min(), xgb_preds.min(), rf_preds.min())
max_val = max(y_test.max(), xgb_preds.max(), rf_preds.max())

# --- Panel 1: XGBoost ---
sns.scatterplot(x=y_test, y=xgb_preds, ax=axes[0], alpha=0.2, color="#1f77b4", edgecolor=None)
axes[0].plot([min_val, max_val], [min_val, max_val], '--r', linewidth=2, label="Predicción Perfecta")
axes[0].set_title('XGBoost: Temperatura Real vs Predicha', fontsize=14, pad=10)
axes[0].set_xlabel('Temperatura Real (°C)', fontsize=12)
axes[0].set_ylabel('Temperatura Predicha (°C)', fontsize=12)
axes[0].legend()

# --- Panel 2: Random Forest ---
sns.scatterplot(x=y_test, y=rf_preds, ax=axes[1], alpha=0.2, color="#2ca02c", edgecolor=None)
axes[1].plot([min_val, max_val], [min_val, max_val], '--r', linewidth=2, label="Predicción Perfecta")
axes[1].set_title('Random Forest (Ligero): Temperatura Real vs Predicha', fontsize=14, pad=10)
axes[1].set_xlabel('Temperatura Real (°C)', fontsize=12)
axes[1].legend()

# Ajustar el layout y mostrar
plt.tight_layout()
plt.show()